# Bi-GRU-LSTM-CNN for Vietnamese Targeted Hate Speech Detection
**Dataset:** ViTHSD (Vietnamese Targeted Hate Speech Dataset)  
**Task:** Multi-label hate speech detection with target categories  
**Model:** Hybrid Bi-GRU + LSTM + CNN architecture

## 1. Install & Import Libraries

In [ ]:
!pip install -q underthesea pyvi scikit-learn seaborn

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, hamming_loss
)
from sklearn.preprocessing import MultiLabelBinarizer

import re
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Data Loading
ViTHSD dataset — same splits as the paper (Train: 7,000 | Dev: 1,201 | Test: 1,800)

In [ ]:
# ─── Adjust paths to your actual ViTHSD files ───
TRAIN_PATH = '/kaggle/input/vithsd/train.csv'   # or .json
DEV_PATH   = '/kaggle/input/vithsd/dev.csv'
TEST_PATH  = '/kaggle/input/vithsd/test.csv'

# Expected CSV columns: 'comment', and label columns like
# 'Individuals', 'Groups', 'Race', 'Politics', 'Religion'
# with values in {0=Normal, 1=Clean, 2=Offensive, 3=Hate}
# OR a 'labels' column containing a list string e.g. "[Individuals#Hate, Groups#Offensive]"

def load_data(path):
    df = pd.read_csv(path)
    print(f'Loaded {path}: {len(df)} rows | columns: {list(df.columns)}')
    return df

train_df = load_data(TRAIN_PATH)
dev_df   = load_data(DEV_PATH)
test_df  = load_data(TEST_PATH)
train_df.head(3)

## 3. Label Schema & Preprocessing

In [ ]:
# ── Label definitions (11 labels — same as paper) ──
TARGETS   = ['Individuals', 'Groups', 'Race', 'Politics', 'Religion']
LEVELS    = ['Offensive', 'Hate']
LABEL_SET = ['Normal'] + [f'{t}#{l}' for t in TARGETS for l in LEVELS]
NUM_LABELS = len(LABEL_SET)   # 11
LABEL2IDX  = {l: i for i, l in enumerate(LABEL_SET)}
print('Labels:', LABEL_SET)

def parse_labels(label_str):
    """
    Parse label string like "['Individuals#Hate', 'Groups#Offensive']"
    or comma-separated 'Individuals#Hate,Groups#Offensive'
    Returns a binary vector of length NUM_LABELS.
    Adapt this function to match your actual CSV format.
    """
    vec = np.zeros(NUM_LABELS, dtype=np.float32)
    if isinstance(label_str, str):
        # Strip brackets / quotes
        label_str = re.sub(r"[\[\]'\"]", '', label_str)
        labels = [l.strip() for l in label_str.split(',') if l.strip()]
    elif isinstance(label_str, list):
        labels = label_str
    else:
        labels = []

    if not labels or labels == ['']:
        vec[LABEL2IDX['Normal']] = 1.0
        return vec

    for lbl in labels:
        if lbl in LABEL2IDX:
            vec[LABEL2IDX[lbl]] = 1.0
    return vec

# ── Quick text normalization for Vietnamese ──
def normalize_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'http\S+', ' ', text)          # remove URLs
    text = re.sub(r'@\w+', ' ', text)             # remove mentions
    text = re.sub(r'[^\w\s]', ' ', text)          # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for df in [train_df, dev_df, test_df]:
    df['clean_text'] = df['comment'].apply(normalize_text)
    df['label_vec']  = df['labels'].apply(parse_labels)

print('Sample:', train_df[['comment','label_vec']].iloc[0])

## 4. Vocabulary & Tokenization

In [ ]:
MAX_VOCAB = 30_000
MAX_LEN   = 128
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

# Build vocabulary from train set only
word_counts = Counter()
for text in train_df['clean_text']:
    word_counts.update(text.split())

vocab = [PAD_TOKEN, UNK_TOKEN] + [w for w, _ in word_counts.most_common(MAX_VOCAB - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
print(f'Vocabulary size: {VOCAB_SIZE}')

def tokenize(text, max_len=MAX_LEN):
    tokens = text.split()[:max_len]
    ids    = [word2idx.get(t, word2idx[UNK_TOKEN]) for t in tokens]
    # Pad
    ids += [word2idx[PAD_TOKEN]] * (max_len - len(ids))
    mask = [1 if t != word2idx[PAD_TOKEN] else 0 for t in ids]
    return ids, mask

# Quick test
sample_ids, sample_mask = tokenize(train_df['clean_text'].iloc[0])
print('Token IDs (first 10):', sample_ids[:10])

## 5. Dataset & DataLoader

In [ ]:
class HateSpeechDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts  = df['clean_text'].tolist()
        self.labels = np.stack(df['label_vec'].values)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, mask = tokenize(self.texts[idx], self.max_len)
        return (
            torch.tensor(ids,  dtype=torch.long),
            torch.tensor(mask, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

BATCH_SIZE = 32

train_ds = HateSpeechDataset(train_df)
dev_ds   = HateSpeechDataset(dev_df)
test_ds  = HateSpeechDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Dev: {len(dev_loader)} | Test: {len(test_loader)}')

## 6. Model Architecture: Bi-GRU + LSTM + CNN

In [ ]:
class BiGRU_LSTM_CNN(nn.Module):
    """
    Hybrid architecture:
      Embedding → CNN (local n-gram features)
               → Bi-GRU (long-range context)
               → LSTM (sequential memory)
               → Attention pooling
               → Multi-label classifier
    """
    def __init__(
        self,
        vocab_size,
        embed_dim      = 128,
        num_filters    = 128,
        kernel_sizes   = [2, 3, 4],
        gru_hidden     = 128,
        lstm_hidden    = 128,
        num_labels     = 11,
        dropout        = 0.3,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # ── CNN branch: multi-scale n-gram convolutions ──
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k, padding=k//2)
            for k in kernel_sizes
        ])
        cnn_out_dim = num_filters * len(kernel_sizes)

        # ── Bi-GRU ──
        self.bigru = nn.GRU(
            embed_dim, gru_hidden,
            bidirectional=True, batch_first=True, dropout=dropout
        )

        # ── LSTM on top of Bi-GRU ──
        self.lstm = nn.LSTM(
            gru_hidden * 2, lstm_hidden,
            batch_first=True, dropout=dropout
        )

        # ── Attention ──
        self.attn = nn.Linear(lstm_hidden, 1)

        # ── Final classifier ──
        fusion_dim = cnn_out_dim + lstm_hidden
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
        # input_ids: (B, L)
        x = self.embed(input_ids)          # (B, L, E)
        x = self.dropout(x)

        # ── CNN ──
        xc = x.permute(0, 2, 1)           # (B, E, L)
        cnn_feats = []
        for conv in self.convs:
            c = F.relu(conv(xc))           # (B, F, L)
            c = F.adaptive_max_pool1d(c, 1).squeeze(-1)  # (B, F)
            cnn_feats.append(c)
        cnn_out = torch.cat(cnn_feats, dim=-1)  # (B, F*K)

        # ── Bi-GRU ──
        gru_out, _ = self.bigru(x)         # (B, L, H*2)

        # ── LSTM ──
        lstm_out, _ = self.lstm(gru_out)   # (B, L, H)

        # ── Attention pooling ──
        attn_w = self.attn(lstm_out)       # (B, L, 1)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            attn_w = attn_w.masked_fill(mask == 0, -1e9)
        attn_w = torch.softmax(attn_w, dim=1)
        seq_repr = (lstm_out * attn_w).sum(dim=1)  # (B, H)

        # ── Fuse CNN + sequence ──
        fused = torch.cat([cnn_out, seq_repr], dim=-1)  # (B, F*K+H)
        logits = self.classifier(self.dropout(fused))   # (B, num_labels)
        return logits


model = BiGRU_LSTM_CNN(vocab_size=VOCAB_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 7. Class-Weighted Loss (handles imbalance)

In [ ]:
# Compute per-label positive weights for BCEWithLogitsLoss
all_labels = np.stack(train_df['label_vec'].values)
pos_counts  = all_labels.sum(axis=0)                          # (11,)
neg_counts  = len(all_labels) - pos_counts
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)
print('pos_weights:', np.round(pos_weights.cpu().numpy(), 2))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

## 8. Training

In [ ]:
EPOCHS    = 20
LR        = 2e-3
THRESHOLD = 0.5

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, 'max', patience=3, factor=0.5, verbose=True)

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0
    all_preds, all_targets = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            logits = model(ids, mask)
            loss   = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > THRESHOLD).float().cpu().numpy()
            all_preds.append(preds)
            all_targets.append(labels.cpu().numpy())

    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    f1_micro = f1_score(all_targets, all_preds, average='micro', zero_division=0) * 100
    f1_macro = f1_score(all_targets, all_preds, average='macro', zero_division=0) * 100
    avg_loss = total_loss / len(loader)
    return avg_loss, f1_micro, f1_macro

history = {'train_loss':[], 'dev_f1_micro':[], 'dev_f1_macro':[]}
best_f1  = 0
CKPT     = 'bigru_lstm_cnn_best.pt'

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, _       = run_epoch(train_loader, training=True)
    dev_loss, dev_f1, dev_mac = run_epoch(dev_loader,   training=False)
    scheduler.step(dev_f1)

    history['train_loss'].append(tr_loss)
    history['dev_f1_micro'].append(dev_f1)
    history['dev_f1_macro'].append(dev_mac)

    print(f'Epoch {epoch:02d} | Train Loss: {tr_loss:.4f} | '
          f'Train F1-Mi: {tr_f1:.2f}% | Dev F1-Mi: {dev_f1:.2f}% | Dev F1-Ma: {dev_mac:.2f}%')

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.state_dict(), CKPT)
        print(f'  → Saved checkpoint (best dev F1-Mi: {best_f1:.2f}%)')

print(f'\nBest Dev F1-Micro: {best_f1:.2f}%')

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].set(title='Training Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()

axes[1].plot(history['dev_f1_micro'], label='Dev F1-Micro', color='tab:blue')
axes[1].plot(history['dev_f1_macro'], label='Dev F1-Macro', color='tab:orange')
axes[1].set(title='Dev F1 Scores', xlabel='Epoch', ylabel='F1 (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('bigru_lstm_cnn_training.png', dpi=150)
plt.show()

## 10. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for ids, mask, labels in test_loader:
        ids, mask = ids.to(device), mask.to(device)
        logits    = model(ids, mask)
        preds     = (torch.sigmoid(logits) > THRESHOLD).float().cpu().numpy()
        all_preds.append(preds)
        all_targets.append(labels.numpy())

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_targets)

metrics = {
    'F1-Micro':        f1_score(y_true, y_pred, average='micro',   zero_division=0) * 100,
    'F1-Macro':        f1_score(y_true, y_pred, average='macro',   zero_division=0) * 100,
    'F1-Samples':      f1_score(y_true, y_pred, average='samples', zero_division=0) * 100,
    'Precision-Micro': precision_score(y_true, y_pred, average='micro', zero_division=0) * 100,
    'Precision-Macro': precision_score(y_true, y_pred, average='macro', zero_division=0) * 100,
    'Recall-Micro':    recall_score(y_true, y_pred, average='micro',    zero_division=0) * 100,
    'Recall-Macro':    recall_score(y_true, y_pred, average='macro',    zero_division=0) * 100,
    'Hamming Loss':    hamming_loss(y_true, y_pred),
}

print('='*50)
print('TEST SET RESULTS — Bi-GRU-LSTM-CNN')
print('='*50)
for k, v in metrics.items():
    fmt = f'{v:.4f}' if 'Hamming' in k else f'{v:.2f}%'
    print(f'  {k:<22}: {fmt}')
print('='*50)

## 11. Per-Label F1 Analysis

In [ ]:
per_label_f1 = f1_score(y_true, y_pred, average=None, zero_division=0) * 100

label_df = pd.DataFrame({
    'Label':    LABEL_SET,
    'F1 (%)':   per_label_f1,
    'Support':  y_true.sum(axis=0).astype(int)
}).sort_values('F1 (%)', ascending=False)

print(label_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(label_df['Label'], label_df['F1 (%)'], color='steelblue')
ax.set(xlabel='F1 (%)', title='Per-Label F1 Score — Bi-GRU-LSTM-CNN (Test Set)')
ax.axvline(x=metrics['F1-Micro'], color='red', linestyle='--', label=f"F1-Micro {metrics['F1-Micro']:.2f}%")
ax.legend()
for bar, v in zip(bars, label_df['F1 (%)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{v:.1f}', va='center')
plt.tight_layout()
plt.savefig('bigru_lstm_cnn_per_label_f1.png', dpi=150)
plt.show()

## 12. Comparison with Paper Baselines

In [ ]:
comparison = pd.DataFrame({
    'Model':       ['HARE (paper)', 'Qwen Vanilla (paper)', 'PhoBERT (paper)', 'Flan-T5 (paper)', 'Bi-GRU-LSTM-CNN (ours)'],
    'F1-Micro':    [60.26, 59.00, 54.12, 46.84, metrics['F1-Micro']],
    'F1-Macro':    [30.35, 33.10, 25.86, 13.11, metrics['F1-Macro']],
    'Prec-Micro':  [63.47, 61.00, 56.20, 48.10, metrics['Precision-Micro']],
    'Rec-Micro':   [57.35, 56.00, 53.10, 45.20, metrics['Recall-Micro']],
})
comparison = comparison.set_index('Model')
print(comparison.round(2).to_string())

ax = comparison[['F1-Micro','F1-Macro','Prec-Micro','Rec-Micro']].plot(
    kind='bar', figsize=(12, 5), rot=30, colormap='tab10'
)
ax.set(title='Model Comparison on ViTHSD Test Set', ylabel='Score (%)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('bigru_lstm_cnn_comparison.png', dpi=150)
plt.show()

## 13. Inference Demo

In [ ]:
def predict(text, threshold=THRESHOLD):
    model.eval()
    clean  = normalize_text(text)
    ids, mask = tokenize(clean)
    ids_t  = torch.tensor([ids],  dtype=torch.long).to(device)
    mask_t = torch.tensor([mask], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(ids_t, mask_t)
        probs  = torch.sigmoid(logits).cpu().numpy()[0]
    results = [(LABEL_SET[i], float(probs[i])) for i in range(NUM_LABELS) if probs[i] > threshold]
    if not results:
        results = [('Normal', float(probs[0]))]
    return sorted(results, key=lambda x: -x[1])

# Test samples from paper
test_samples = [
    'Thằng A ngu thế cũng làm ca sĩ, bọn fan của nó thì cũng súc vật không kém',
    'Lũ ba que xỏ lá',
    'Optimusthree với 99% sức mạnh',
    'Hôm nay trời đẹp quá!',
]

for txt in test_samples:
    preds = predict(txt)
    print(f'Text: {txt}')
    print(f'  → Predictions: {preds}\n')